# Group comparison of stimulation responses

This notebook compares the responses to stimulation between groups of participants (here, younger and older
adults), following the analysis of the BigStim study:

1. for every participant, cut 30 s epochs (-15 to +15 s) around the stimulations made during N2 and N3 sleep, in
   the slow band (slow-wave activity, SWA), the sigma envelope (spindles) and the broadband EEG;
2. reject artifacts, subtract a baseline (-2.5 to -0.5 s before the stimulation) and average per participant;
3. compare the groups, here on the amplitude of the slow and sigma responses at the moment of the stimulation.

**How to use it:** list the participants in the configuration cell, then run all the cells from the top. With an
empty table, the notebook writes and analyses six synthetic participants.

## 1. Configuration

In [ ]:
from pathlib import Path

import pandas as pd

from portiloop_analysis import EpochSettings

# One row per participant. Columns:
#   participant  identifier used in the output files
#   age_group    group compared in the figures (e.g. "young", "older")
#   condition    any other grouping you want to keep in the table (e.g. targeted rhythm)
#   recording    .csv file of the night
#   hypnogram    sleep scoring (CSV with columns Annotation and Duration in s), or "" if none
#   format       "current" (Portiloop software >= 0.2) or "legacy" (older Portiloop recordings)
participants = pd.DataFrame([
    # {"participant": "P52", "age_group": "young", "condition": "spindle",
    #  "recording": "/data/BSP_P52/EEG/PORTILOOP/BSP_P52_STIMON.csv",
    #  "hypnogram": "/data/BSP_P52/EEG/BSP_P52_STIMON_annotations.txt", "format": "legacy"},
], columns=["participant", "age_group", "condition", "recording", "hypnogram", "format"])

channel = "Cz"
montage = "default"           # for "current" recordings (see slow_and_sigma_response.ipynb)
group_order = ["young", "older"]
output_dir = Path("group_results")

# Whole-night cleaning of the BigStim analysis: remove off-head periods, jumps larger than 222 uV, and stretches
# of good signal shorter than 5 minutes (see portiloop_analysis.bad_sections).
use_bad_sections = True

settings = EpochSettings(
    window_s=15,                       # epochs from -15 to +15 s
    notch_hz=60,                       # mains frequency
    broadband=(0.5, 30),               # band of the ERP and of the artifact check
    slow_band=(0.5, 4),                # the original filtered 0.1 to 4 Hz after 0.5 to 30 Hz, i.e. 0.5 to 4 Hz
    max_ptp_uv=None,
    max_abs_uv=250 / 2.35,             # reject epochs whose broadband EEG exceeds +/- 106 uV
    jump_limit_uv=None,                # jumps are handled by bad_sections above
    slow_baseline=(-2.5, -0.5),
    sigma_baseline=(-2.5, -0.5),
    broadband_baseline=(-2.5, -0.5),
    min_interval_s=2.0,                # markers less than 2 s after the previous one are duplicates
    stages=("N2", "N3"),               # needs a hypnogram, or a legacy file with sorted_stim; None: all stages
)

## 2. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from portiloop_analysis import (bad_sections, event_locked_responses, load_hypnogram, load_legacy_recording,
                                load_recording, make_demo_recording, plot_groups, value_at)

## 3. Demo participants

Without participants in the configuration, six synthetic recordings are written in `demo_data/`: three "young"
participants with large responses and three "older" participants with smaller ones.

In [ ]:
if participants.empty:
    rows = []
    for i in range(6):
        young = i < 3
        csv = make_demo_recording("demo_data", name=f"demo{i + 1:02d}", duration_min=20, seed=10 + i,
                                  slow_wave_uv=60 if young else 35, spindle_uv=12 if young else 7, artifacts=False)
        rows.append({"participant": f"demo{i + 1:02d}", "age_group": "young" if young else "older",
                     "condition": ["spindle", "sleep"][i % 2], "recording": str(csv),
                     "hypnogram": str(csv.with_name(csv.stem + "_hypnogram.csv")), "format": "current"})
    participants = pd.DataFrame(rows)
participants[["participant", "age_group", "condition", "format"]]

## 4. Responses of every participant

For each participant: load the recording and its sleep scoring, clean the night, cut and check the epochs, and save
them in `output_dir/<participant>_responses.npz` (time axis, kept epochs of every band, and the status of every
stimulation marker). The table counts the markers kept and the reasons of rejection.

In [ ]:
def analyse_participant(row):
    if row.format == "legacy":
        recording = load_legacy_recording(row.recording, channel=channel)
    else:
        recording = load_recording(row.recording, channel=channel, montage=montage, verbose=False)
    if isinstance(row.hypnogram, str) and row.hypnogram:
        recording.stages = load_hypnogram(row.hypnogram, recording.fs, recording.n_samples)
    if use_bad_sections:
        good = bad_sections(recording.eeg, recording.fs)
        recording.eeg = np.where(good, recording.eeg, np.nan)
    return event_locked_responses(recording, settings)


output_dir.mkdir(exist_ok=True)
responses = {}
counts = []
for row in participants.itertuples():
    result = analyse_participant(row)
    responses[row.participant] = result
    np.savez_compressed(output_dir / f"{row.participant}_responses.npz", time=result.time,
                        events_sample=result.events["sample"].to_numpy(),
                        events_status=result.events["status"].to_numpy().astype(str),
                        **{f"{band}_epochs": epochs for band, epochs in result.epochs.items()})
    counts.append({"participant": row.participant, **result.rejection_summary().to_dict()})

counts = pd.DataFrame(counts).fillna(0).set_index("participant").astype(int)
counts

## 5. Values at the moment of the stimulation

For each participant, the average slow-band and sigma-envelope responses are read at t = 0, the moment of the
stimulation, after baseline correction. Averages with values above 1000 uV (artifacts that passed the checks) are
discarded.

**Caution:** the filters are zero-phase (they run forwards and backwards), so they spread a response a little backwards in time: part of a large evoked wave leaks into t = 0 and the samples just before it. Differences between groups at t = 0 can therefore come from the response that follows the sound. Choose your measurement windows with this in mind, before looking at the data (see tutorial 09).

In [ ]:
rows = []
for row in participants.itertuples():
    result = responses[row.participant]
    values = {"participant": row.participant, "age_group": row.age_group, "condition": row.condition,
              "epochs": result.n_kept}
    for band in ("slow", "sigma"):
        average = result.mean(band)
        if np.nanmax(np.abs(average)) > 1000:
            print(f"{row.participant}: {band} average above 1000 uV, discarded")
            average = np.full_like(average, np.nan)
        values[f"{band}_at_stimulation_uV"] = value_at(result.time, average, 0.0)
    rows.append(values)
values = pd.DataFrame(rows)
values.to_csv(output_dir / "values_at_stimulation.csv", index=False)
values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
plot_groups(axes[0], values, "sigma_at_stimulation_uV", "age_group", order=group_order,
            ylabel="Spindle envelope amplitude (µV)")
plot_groups(axes[1], values, "slow_at_stimulation_uV", "age_group", order=group_order,
            ylabel="Slow-wave activity amplitude (µV)")
fig.savefig(output_dir / "values_at_stimulation.pdf")
plt.show()

## 6. Group average responses

Average of the participants' average responses, +/- 1 SEM across participants, for every group.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, layout="constrained")
for group, color in zip(group_order, ["#F4A261", "#2166AC"]):
    members = values.loc[values["age_group"] == group, "participant"]
    for ax, band in zip(axes, ("slow", "sigma")):
        averages = np.array([responses[p].mean(band) for p in members if responses[p].n_kept])
        if len(averages) == 0:
            continue
        time = responses[members.iloc[0]].time
        mean = averages.mean(axis=0)
        sem = averages.std(axis=0, ddof=1) / np.sqrt(len(averages)) if len(averages) > 1 else 0 * mean
        ax.plot(time, mean, color=color, label=f"{group} (n = {len(averages)})")
        ax.fill_between(time, mean - sem, mean + sem, color=color, alpha=0.25, linewidth=0)
for ax, label in zip(axes, ["Slow-band EEG (µV)", "Sigma envelope (µV)"]):
    ax.axvline(0, color="red", linestyle="--")
    ax.axvspan(*settings.slow_baseline, color="0.85", linewidth=0)
    ax.set_ylabel(label)
    ax.legend(frameon=False)
axes[-1].set_xlim(-5, 5)
axes[-1].set_xlabel("Time relative to the stimulation (s)")
fig.savefig(output_dir / "group_averages.pdf")
plt.show()